# Ingredient Clustering with Sentence Embeddings

**Goal**: Automatically group semantically similar ingredients using vector embeddings and hierarchical clustering. This is a foundational building block for recipe recommendation systems, food ontologies, and smart ingredient substitution.

### Approach
1. **Embed** each ingredient using a pretrained sentence transformer (`nomic-embed-text-v1.5`)
2. **Cluster** the embeddings with agglomerative hierarchical clustering
3. **Evaluate** using silhouette score and Davies-Bouldin index
4. **Visualize** the clusters via PCA projection

---
## 1. Setup & Imports

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
import nltk
import numpy as np
import matplotlib.pyplot as plt

nltk.download('wordnet', quiet=True)

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", device="cpu")

---
## 2. Ingredient Data

A diverse list covering produce, mushrooms, fruits, proteins, dairy, oils, spices, and pantry staples — including some intentionally similar items (e.g., *"tomato"* vs *"cherry tomato"*, *"mushrooms"* vs *"shiitake"*) to test the clusterer's ability to distinguish fine-grained categories.

In [ ]:
raw_ingredients = [
    # Produce & Herbs
    "tomato", "cherry tomato", "bell pepper", "red pepper", "jalapeno",
    "chili", "cucumber", "lettuce", "pumpkin", "potato", "sweet potato",
    "asparagus", "garlic", "onions", "green onions", "parsley", "basil",

    # Mushrooms
    "shiitake", "champignons", "enoki mushroom", "oyster mushroom", "mushrooms",

    # Fruits
    "cherry", "cherries", "melon", "water melon",

    # Proteins & Seafood
    "white fish",

    # Dairy & Cheeses
    "milk", "whole milk", "cream", "whipped cream", "cream cheese",
    "gouda", "blue cheese", "butter",

    # Oils, Pastes & Nuts
    "oil", "truffle oil", "tahini", "peanut butter", "peanuts",
    "sesame", "black sesame",

    # Pantry, Liquids & Spices
    "water", "stock", "vegetable broth", "fish stock", "mustard",
    "black pepper", "white pepper", "pepper corn", "salt",

    # New Interesting Additions
    "miso paste", "soy sauce", "gochujang", "kimchi", "coconut milk",
    "smoked paprika", "hot honey", "yuzu juice",
]

---
## 3. Preprocessing: Lemmatization

Normalize plural forms (e.g., *onions* → *onion*, *cherries* → *cherry*) so the embedding layer sees canonical forms.

In [ ]:
lemmatizer = nltk.stem.WordNetLemmatizer()
ingredients = [lemmatizer.lemmatize(str.lower(i)) for i in raw_ingredients]

---
## 4. Embedding

Convert each ingredient into a 768-dimensional vector using `nomic-embed-text-v1.5`. Embeddings are L2-normalized so cosine similarity is equivalent to dot-product similarity.

In [ ]:
X = model.encode(
    ingredients,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f"Embedding dimension: {X.shape[1]}")

---
## 5. Hierarchical Agglomerative Clustering

Using **average linkage** with a **distance threshold of 0.99** — automatically determines the number of clusters by cutting the dendrogram where merge distances exceed the threshold.

Alternative approaches tried (commented out):
- Ward linkage with `text-embedding-3-large` (OpenAI)
- HDBSCAN with cosine metric

In [ ]:
clustering = AgglomerativeClustering(
    linkage='average',
    n_clusters=None,
    distance_threshold=0.52,
    metric='cosine'
).fit(X)

print(f"Number of clusters: {len(set(clustering.labels_))}")

---
## 6. Cluster Contents

In [ ]:
grouped = {}
for ingredient, label in zip(ingredients, clustering.labels_):
    grouped.setdefault(label, []).append(ingredient)

for label in sorted(grouped):
    print(f"Cluster {label}: {grouped[label]}")

---
## 7. Evaluation & Visualization

Two complementary metrics:
- **Silhouette Score** (higher is better, range [-1, 1]): measures how similar an object is to its own cluster vs. other clusters.
- **Davies-Bouldin Index** (lower is better): average similarity between each cluster and its most similar one.

PCA reduces the 768-d embeddings to 2-d for visualization. Cluster centroids are marked with black **X** markers.

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
labels = clustering.labels_

# ---- metrics ----
sil_score = silhouette_score(X, labels)
db_score = davies_bouldin_score(X, labels)

print(f"Silhouette Score:  {sil_score:.3f}")
print(f"Davies-Bouldin Index: {db_score:.3f}")

# ---- centroids in PCA space ----
unique_labels = np.unique(labels)
centroids = np.array([
    X_2d[labels == l].mean(axis=0)
    for l in unique_labels
])

# ---- plot ----
plt.figure(figsize=(11, 8))

scatter = plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=labels, cmap='tab10', s=80,
    alpha=0.75, edgecolors='white', linewidth=0.5,
)

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    c='black', s=200, marker='X', label='Centroids',
)

for i, l in enumerate(unique_labels):
    plt.text(
        centroids[i, 0], centroids[i, 1],
        f"Cluster {l}", fontsize=11, weight='bold',
        ha='center', va='center',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'),
    )

plt.title(
    f"Ingredient Clusters (PCA Projection)\n"
    f"Silhouette: {sil_score:.2f} | Davies-Bouldin: {db_score:.2f}"
)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid(True, linestyle='--', alpha=0.3)
plt.colorbar(scatter, label="Cluster ID")
plt.legend()
plt.tight_layout()
plt.show()

---
## 8. Inter-Cluster Cosine Similarity

Compute pairwise cosine similarity between cluster centroids to measure how distinct (or similar) each cluster is from every other.
Since embeddings are L2-normalized, cosine similarity reduces to the dot product of centroids.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cluster centroids in the original embedding space
unique_labels = np.unique(clustering.labels_)
centroids = np.array([
    X[clustering.labels_ == l].mean(axis=0)
    for l in unique_labels
])

sim = cosine_similarity(centroids)

# --- human-readable matrix ---
header = "     " + "".join(f"{l:>7d}" for l in unique_labels)
print("Pairwise Cosine Similarity Between Cluster Centroids")
print("=" * (8 + 7 * len(unique_labels)))
print(header)
for i, l1 in enumerate(unique_labels):
    row = "".join(f"{sim[i, j]:>7.3f}" for j in range(len(unique_labels)))
    print(f"  {l1:>2d}  {row}")

# --- top-5 most similar pairs ---
pairs = []
for i in range(len(unique_labels)):
    for j in range(i + 1, len(unique_labels)):
        pairs.append((unique_labels[i], unique_labels[j], sim[i, j]))
pairs.sort(key=lambda x: -x[2])

print("\nTop-5 Most Similar Cluster Pairs:")
for l1, l2, val in pairs[:5]:
    print(f"  Cluster {l1:>2d}  <->  Cluster {l2:>2d}   =   {val:.4f}")

threshold = 0.67
connections = []
for i in range(len(unique_labels)):
    for j in range(i + 1, len(unique_labels)):
        if sim[i, j] > threshold:
            connections.append((int(unique_labels[i]), int(unique_labels[j]), float(sim[i, j])))

if not connections:
    print(f"No centroid pairs with cosine similarity > {threshold}")
else:
    connections.sort(key=lambda x: -x[2])
    print(f"Centroid pairs with cosine similarity > {threshold}:")
    for a, b, v in connections:
        print(f"  Cluster {a:>2d} <-> Cluster {b:>2d}   =   {v:.4f}")

# Find highest similarity for a new ingredient "parmesan"
test_ingredient = "tuna"
test_embedding = model.encode([test_ingredient], normalize_embeddings=True, convert_to_numpy=True)
similarities = cosine_similarity(test_embedding, X)[0]
max_idx = np.argmax(similarities)
max_similarity = similarities[max_idx]

print(f"\nHighest similarity for '{test_ingredient}':")
print(f"  Ingredient: {ingredients[max_idx]}")
print(f"  Similarity: {max_similarity:.4f}")
print(f"  Cluster: {clustering.labels_[max_idx]}")

---
## 9. Hyperparameter Tuning Notes

Various configurations explored during development:

| Embedding Model | Linkage | Metric | Distance Threshold | Silhouette | Davies-Bouldin | Notes |
|---|---|---|---|---|---|---|
| nomic-embed-text-v1.5 | ward | euclidean | 1.10 | 0.18 | 1.32 | |
| nomic-embed-text-v1.5 | average | euclidean | 1.00 | 0.22 | 1.22 | |
| nomic-embed-text-v1.5 | average | euclidean | 0.95 | 0.14 | 0.90 | Too few clusters |
| nomic-embed-text-v1.5 | average | euclidean | 0.97 | 0.14 | 0.90 | |
| nomic-embed-text-v1.5 | HDBSCAN (min=2, cosine) | cosine | — | 0.10 | 1.92 | Many singletons |
| nomic-embed-text-v1.5 | average | cosine | **0.52** | 0.13 | 1.15 | Current config — 19 clusters |
| nomic-embed-text-v1.5 | average | cosine | 0.7 | — | — | Only 1 category |
| nomic-embed-text-v1.5 | average | cosine | 0.6 | — | — | Too few clusters |
| nomic-embed-text-v1.5 | average | cosine | 0.49 | — | — | 20+ categories |

The **average linkage with cosine metric and threshold 0.52** offers the best balance — producing semantically coherent clusters without excessive fragmentation.

---
## 10. Key Takeaways

- **Sentence transformers** produce high-quality ingredient embeddings without task-specific fine-tuning.
- **Agglomerative clustering** with a fixed distance threshold works well for this domain, naturally separating produce, dairy, spices, mushrooms, and condiments.
- The approach successfully groups near-synonyms (*mushrooms* / *champignons*, *cherry* / *cherries*, *bell pepper* / *red pepper*) while keeping distinct categories apart.
- **Open AI text-embedding-3-large** in combination with Agglomerative clustering produced the best results
- Merging clusters based on cosine similarity should not be applied to all clusters, as mostly clusters are merged which already have a lot of ingredients but outliers which should be merged have a lower similarity to all other clusters. This is why the cosine similarity should only be applied to outliers and in same cases should be treated as individual categories, as they might differ totally from all other categories. 
- **Cosine metric is far more effective than default Euclidean** for ingredient clustering. Cosine measures angular similarity between embedding vectors, focusing on the *direction* of the semantic content rather than vector magnitude. Two ingredients can have embeddings with very different magnitudes (e.g., due to length or specificity of the ingredient name) yet point in nearly the same semantic direction — Euclidean distance would incorrectly separate them. Cosine distance aligns naturally with the L2-normalized embeddings used by sentence transformers, making it the correct choice for category discovery.